Take a model and run SHAP on it on some of our binary MMOTU-2D dataset.

In [ ]:
from datasets import load_dataset
from endometriosis.data import DEFAULT_DATASET_PATH, ID2LABEL as id2label, LABEL2ID as label2id, with_image_transform

dataset = load_dataset(DEFAULT_DATASET_PATH)

In [ ]:
import numpy as np 
# Print some info about the dataset
print("Dataset info:")
print(f"Number of samples: {len(dataset['train']) + len(dataset['test'])}")
print("Traaining label distribution:")
labels, counts = np.unique(dataset['train']['label'], return_counts=True)
for label, count in zip(labels, counts):
    print(f"  {label}: {count}")

print("Testing label distribution:")
labels, counts = np.unique(dataset['test']['label'], return_counts=True)
for label, count in zip(labels, counts):
    print(f"  {label}: {count}")

In [ ]:
# Load a fine-tuned model
from endometriosis.models import load_model

loaded = load_model("omnirad", checkpoint="../outputs/checkpoints/Snarcy/OmniRad-small-finetuned-binary-weightedCE-mmotu/checkpoint-80")
model = loaded.model
mean, std = loaded.mean, loaded.std

test_ds = with_image_transform(dataset["test"], loaded.val_transforms)

In [ ]:
# Plot an image
import matplotlib.pyplot as plt
import torch

def show_image(image_tensor, label):
    # As we're using a pre-processed dataset, we need to undo the normalisation
    image_tensor = image_tensor * torch.tensor(std).view(3, 1, 1) + torch.tensor(mean).view(3, 1, 1)

    image = image_tensor.permute(1, 2, 0).numpy()  # Convert from (C, H, W) to (H, W, C)
    plt.imshow(image)
    plt.title(f"Label: {label}")
    plt.axis("off")
    plt.show()

show_image(test_ds[0]["pixel_values"], test_ds[0]["label"])

In [ ]:
# Given an image, get the model's prediction
from endometriosis.inference import predict

test_idx = 0

predicted_label, probabilities = predict(model, test_ds[test_idx]["pixel_values"])
print(f"Predicted label: {id2label[predicted_label]}, Probabilities: {probabilities}, True label: {id2label[test_ds[test_idx]['label']]}")

In [ ]:
# Move model to the GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
# Given a sample, compute SHAP values and overlay (GradientExplainer)
import shap
import numpy as np
import torch
import torch.nn as nn

class HFImageClassifierForSHAP(nn.Module):
    def __init__(self, hf_model):
        super().__init__()
        self.hf_model = hf_model

    def forward(self, pixel_values):
        outputs = self.hf_model(pixel_values=pixel_values)
        return outputs.logits

model.eval()
wrapped_model = HFImageClassifierForSHAP(model)

device = next(model.parameters()).device
wrapped_model.to(device)

# Create a SHAP GradientExplainer (PyTorch backend)
background = torch.zeros((1, 3, 224, 224), device=device)
explainer = shap.GradientExplainer(wrapped_model, background)

test_idx = 221

# Get the image tensor
image_tensor = test_ds[test_idx]["pixel_values"].unsqueeze(0).to(device)

# Compute SHAP values
shap_values = explainer.shap_values(image_tensor)

# Convert image to H, W, C numpy format expected by shap.image_plot
image_np = image_tensor.detach().cpu().squeeze(0)
image_np = image_np * torch.tensor(std).view(3, 1, 1) + torch.tensor(mean).view(3, 1, 1)
image_np = image_np.permute(1, 2, 0).numpy()
image_np = np.clip(image_np, 0, 1)
image_np = np.expand_dims(image_np, axis=0)

# Convert SHAP output to H, W, C layout expected by shap.image_plot
if isinstance(shap_values, list):
    shap_values_hwc = [np.transpose(class_values, (0, 2, 3, 1)) for class_values in shap_values]
elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 4:
    shap_values_hwc = np.transpose(shap_values, (0, 2, 3, 1))
elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 5:
    shap_values_hwc = [np.transpose(shap_values[..., i], (0, 2, 3, 1)) for i in range(shap_values.shape[-1])]
else:
    raise ValueError(f"Unexpected SHAP output format: {type(shap_values)}, shape={getattr(shap_values, 'shape', None)}")

# Use the model's predicted class for explanation
with torch.no_grad():
    outputs = model(pixel_values=image_tensor)
class_idx = int(torch.argmax(outputs.logits, dim=1).item())

# Extract the SHAP map for the chosen class
# shap_values_hwc may be an array shaped (classes, batch, H, W, C)
class_shap = shap_values_hwc[class_idx]

# Plot only the chosen class explanation
shap.image_plot([class_shap], image_np, true_labels=[id2label[test_ds[test_idx]["label"]]], labels=[id2label[class_idx]])

In [ ]:
# Find a sample where the model correctly predicts endometriosis
for test_idx in range(len(test_ds)):
    predicted_label, probabilities = predict(model, test_ds[test_idx]["pixel_values"].to(device))
    true_label = test_ds[test_idx]["label"]
    if predicted_label == true_label == label2id["non-endometriosis"]:
        print(f"Found correctly predicted endometriosis sample at index {test_idx}")